### Loopealsete eraldamine ELME kihist 2019. a rohtse biomassi mudeli jaoks



Iris Luik, 2026

Tartu Ülikool, Geograafia osakond

In [ ]:
# kirjutan üle

import os

os.environ['PROJ_DATA'] = r'C:\Users\irisl\micromamba\envs\geopython2025\Library\share\proj'
os.environ['PROJ_LIB'] = r'C:\Users\irisl\micromamba\envs\geopython2025\Library\share\proj'

In [ ]:
# vajaminevad paketid

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

In [ ]:
# loen terve niitude shp kihi sisse

niidud = gpd.read_file("orig_parandniidud/parandniidud_ELME2_seisund.shp", encoding='latin1')
niidud.head(5)

In [ ]:
# kontrollin niitude arvu tüüpide kaupa

# 280* on loopealsed https://www.loodusveeb.ee/sites/default/files/inline-files/ELME2_LOPPARUANNE_fin_151123.pdf
niidud["pohityyp"].value_counts(dropna=False)

In [ ]:
# jätan alles ainult loopealsed (pohityyp=6280*)

loopealsed = niidud[niidud["pohityyp"]== "6280*"]
loopealsed["pohityyp"].value_counts(dropna=False)

In [ ]:
# jätan alles ainult A ja B seisukorras (seis_komb) loopealsed

loopealsedAjaB = loopealsed[loopealsed["seis_komb"].isin(["A", "B"])]
len(loopealsedAjaB)

In [ ]:
# ühendan kõik alad üheks geomeetriaks ja seejärel jagan multipolügonid eraldi polügonideks

dissolved = loopealsedAjaB.dissolve()
singleparts = dissolved.explode(index_parts=True).reset_index(drop=True)

In [ ]:
# kontrollin saadud loopealsete arvu

len(singleparts)

In [ ]:
# loen sisse KIK2019 kihi
# kasutan seda mudeli AOA leidmiseks 

KIK2019 = pd.read_csv("KIK_data/KIK2019.csv")

gdf = gpd.GeoDataFrame(KIK2019, geometry=gpd.points_from_xy(KIK2019["X"], KIK2019["Y"]), crs="EPSG:4326")

# teisendan Eesti koordinaatsüsteemi
gdf = gdf.to_crs("EPSG:3301")

In [ ]:
# leian punktide ulatuse (bounding box)

xmin, ymin, xmax, ymax = gdf.total_bounds

# loon ulatuse põhjal ristkülikukujulise polügoni
bbox = box(xmin, ymin, xmax, ymax) 

# teen selle geodataframe'i
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs=gdf.crs)

In [ ]:
# lõikan A ja B seisukorras loopealsete kihti mudeli AOA-ga ehk loodud ristkülikuga
singleparts_clipped = gpd.clip(singleparts, bbox_gdf)

In [ ]:
# kontrollin polügonide arvu
len(singleparts_clipped)

In [ ]:
# kontrollin visuaalselt

fig, ax = plt.subplots(figsize=(12, 12))

singleparts.plot(ax=ax, color="orange", edgecolor="orange")
bbox_gdf.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
singleparts_clipped.plot(ax=ax, color="green", edgecolor="green")

plt.show()

In [ ]:
# jätan alles ainult geomeetria ja teen id veeru

singleparts_clipped_geom = singleparts_clipped[['geometry']]
singleparts_clipped_geom['id'] = range(1, len(singleparts_clipped) + 1)

singleparts_clipped_geom.head(5)

In [ ]:
# arvutan pindala

singleparts_clipped_geom["area_m2"] = singleparts_clipped_geom.geometry.area
total = singleparts_clipped_geom["area_m2"].sum()/10000
print(f"Mudeli AOA sisse jäävate heade ja väga heade (A või B) loopealsete pindala: {total:.1f} ha")

In [ ]:
# ekspordin shp failina GEE-s kasutamiseks

singleparts_clipped_geom.to_file('loopealsed_A_B_AOA', driver='ESRI Shapefile')